In [ ]:
# Celda 1 - comprobar que Colab dio una GPU
# El assert de torch NO va aqui: la celda 2 instala el paquete y pip puede
# reemplazar el torch preinstalado de Colab para cumplir los pines. La
# comprobacion real del entorno es la celda 3, ya con lo instalado.
!nvidia-smi

In [ ]:
# Celda 2 - traer el codigo e INSTALAR EL PAQUETE de verdad
# Antes esto era un sys.path.insert mas un `pip install -q pydicom` suelto.
# Con eso los pines de pyproject.toml (torch==2.14.0, torchvision==0.29.0,
# pydicom==3.0.2, ...) NO se aplicaban: Colab entrenaba con las versiones que
# trae preinstaladas y el entorno remoto dejaba de reproducir el local, que es
# justo lo que el spec parrafo 10 exige. Instalar el paquete hace que pip
# resuelva esas dependencias y que `import copiloto_triaje` venga del paquete
# instalado, no de una ruta pegada a mano.
import pathlib, zipfile
from google.colab import files

# Opcion A (mientras el repo sea privado): comprimir la carpeta
# copiloto-triaje en el equipo y subirla aqui. Pesa unos 100 KB.
subidos = files.upload()            # seleccionar copiloto-triaje.zip
nombre = next(iter(subidos))
with zipfile.ZipFile(nombre) as z:
    z.extractall('/content')
RAIZ = pathlib.Path('/content/copiloto-triaje')

# Opcion B (cuando el repo sea publico, hito H8): comentar lo de arriba y usar
# !git clone https://github.com/<usuario>/copiloto-triaje.git /content/copiloto-triaje

!pip install -q "{RAIZ}"
print('paquete instalado desde', RAIZ)

In [ ]:
# Celda 3 - verificar el entorno DESPUES de instalar
# Si pip tuvo que reemplazar el torch preinstalado de Colab para cumplir el
# pin, el runtime sigue con el torch viejo cargado en memoria: hay que hacer
# Entorno de ejecucion > Reiniciar sesion y volver a ejecutar DESDE ESTA
# CELDA (no hace falta repetir la subida). Los asserts lo detectan.
import torch, copiloto_triaje
print('torch', torch.__version__, '| cuda', torch.version.cuda)
print('copiloto_triaje desde', copiloto_triaje.__file__)
assert '/content/copiloto-triaje/src' not in copiloto_triaje.__file__, \
    'copiloto_triaje se importa del arbol de fuentes, no del paquete instalado'
assert torch.cuda.is_available(), (
    'Sin GPU. Si acabas de instalar el paquete y pip reemplazo torch, reinicia '
    'el entorno de ejecucion y vuelve a ejecutar desde esta celda. Si no, '
    'Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Celda 4 - Drive para lo que debe sobrevivir, y descarga del dataset desde Kaggle
import os, pathlib
from google.colab import drive, userdata

drive.mount('/content/drive')
PERSISTENTE = pathlib.Path('/content/drive/MyDrive/copiloto-triaje')
PERSISTENTE.mkdir(parents=True, exist_ok=True)

# Secretos de Colab (icono de la llave): KAGGLE_USERNAME y KAGGLE_KEY
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!pip install -q kaggle
!kaggle competitions download -c rsna-pneumonia-detection-challenge -p /content/datos
ZIP = '/content/datos/rsna-pneumonia-detection-challenge.zip'
print('descargado:', pathlib.Path(ZIP).stat().st_size / 1e9, 'GB')

In [ ]:
# Celda 5 - preparar los datos: zip -> PNG -> manifiestos (dentro de Colab)
# Son unos 35 minutos. Revisa la salida: si aparece una linea AVISO, hubo
# imagenes convertidas que no entraron en los manifiestos (spec parrafo 8).
!python {RAIZ}/scripts/preparar_datos.py --zip {ZIP} --raiz /content/trabajo

In [ ]:
# Celda 6 - entrenamiento completo con GPU, checkpoints en Drive
import torch, pandas as pd, pathlib
from torch.utils.data import DataLoader
from copiloto_triaje.config import Config
from copiloto_triaje.datos.dataset import DatasetRadiografias
from copiloto_triaje.modelo.arquitectura import construir_modelo
from copiloto_triaje.modelo.entrenamiento import entrenar

# Los valores por defecto de Config ya son los de la SEGUNDA corrida
# (320 px, DenseNet121, 4+20 epocas, ajuste fino completo, planificador
# coseno). Ver docs/mejoras-2026-09-21.md para la justificacion medida.
# El lote baja a 24 porque 320 px y DenseNet121 pesan mas en memoria que
# 224 px y MobileNetV3; si la GPU da error de memoria, bajalo a 16.
cfg = Config(raiz_datos=pathlib.Path('/content/trabajo'),
             zip_rsna=pathlib.Path(ZIP))
print('configuracion:', cfg.arquitectura, '|', cfg.tam_imagen, 'px | lote',
      cfg.lote, '|', cfg.epocas_cabeza, '+', cfg.epocas_ajuste, 'epocas')
dispositivo = torch.device('cuda')
DIR_CHECKPOINTS = PERSISTENTE / 'checkpoints'   # en Drive: sobrevive a la caida

def cargador(nombre, entrenamiento):
    df = pd.read_csv(cfg.dir_manifiestos / f'{nombre}.csv')
    ds = DatasetRadiografias(df, cfg.dir_png, cfg.tam_imagen, entrenamiento)
    # num_workers=8, no 2: los PNG de RSNA son de 1024x1024 y decodificarlos y
    # redimensionarlos domina el tiempo de epoca. Medido: ~2,7 min por epoca
    # solo de carga con 2 procesos frente a ~0,7 min con 8, con la GPU parada
    # el resto del tiempo. pin_memory acelera ademas la copia a la GPU.
    return DataLoader(ds, batch_size=cfg.lote, shuffle=entrenamiento,
                      num_workers=8, pin_memory=True)

modelo = construir_modelo(cfg.arquitectura, congelar_backbone=True,
                          preentrenado=True)

# REANUDAR=True: si Colab se cayo a mitad del entrenamiento, se retoma desde
# 'ultimo.pt' -pesos, optimizador, epoca, fase, historial y contadores- en vez
# de empezar de cero (spec parrafo 8). Con los checkpoints en Drive sobreviven
# a la caida. En un entrenamiento nuevo da igual: si no hay checkpoint,
# arranca desde el principio sin quejarse.
REANUDAR = True
resultado = entrenar(modelo, cargador('train', True), cargador('val', False),
                     cfg, dispositivo, DIR_CHECKPOINTS, reanudar=REANUDAR)
print('mejor epoca:', resultado['mejor_epoca'],
      '| recall en validacion:', resultado['mejor_recall'],
      '| AUC:', resultado['mejor_auc'])
print('pesos limpios en CPU para SP2:', resultado['ruta_pesos_cpu'])

In [ ]:
# Celda 7 - umbral en VALIDACION, evaluacion UNICA en prueba, reporte a Drive
import json
from copiloto_triaje.modelo.evaluacion import elegir_umbral, metricas_en_umbral
from copiloto_triaje.modelo.entrenamiento import evaluar_epoca
from copiloto_triaje.modelo.reporte import escribir_reporte_markdown, guardar_curva_roc

# `modelo` ya vuelve de entrenar() restaurado a los pesos de su MEJOR
# epoca, no a los de la ultima: no hace falta -ni hay que anadir- una
# recarga manual del checkpoint aqui.
y_val, s_val = evaluar_epoca(modelo, cargador('val', False), dispositivo)
# El recall minimo entra en la regla: sin el, la version anterior conservaba
# el punto de especificidad objetivo aunque incumpliera el recall minimo,
# teniendo disponible otro que cumplia AMBOS minimos.
umbral, m_val, paso = elegir_umbral(y_val, s_val, cfg.especificidad_objetivo,
                                    cfg.especificidad_minima, cfg.recall_objetivo,
                                    cfg.recall_minimo)
print('umbral elegido en validacion:', umbral, '| regla aplicada:', paso)

# El conjunto de prueba se toca UNA sola vez, con el umbral ya congelado.
# metricas_en_umbral, no metricas_binarias: es lo que hace que el AUC del
# conjunto de PRUEBA quede reportado. Es uno de los cuatro criterios de
# aceptacion del spec parrafo 2 y con metricas_binarias era inreportable.
y_test, s_test = evaluar_epoca(modelo, cargador('test', False), dispositivo)
m_test = metricas_en_umbral(y_test, s_test, umbral)
print(json.dumps(m_test, indent=2, ensure_ascii=False))

(PERSISTENTE / 'reporte_metricas.json').write_text(
    json.dumps({'umbral': umbral, 'regla': paso, 'validacion': m_val,
                'prueba': m_test, 'historial': resultado['historial']},
               indent=2, ensure_ascii=False), encoding='utf-8')

# Reporte legible y curva ROC: los dos son entregables del spec (parrafos 7 y
# 12). La logica vive en el paquete, no aqui, para que lo que se prueba en
# local sea exactamente lo que se ejecuta en Colab.
ruta_roc = guardar_curva_roc(y_test, s_test, PERSISTENTE / 'curva_roc.svg',
                             umbral=umbral)
ruta_md = escribir_reporte_markdown(
    PERSISTENTE / 'reporte_metricas.md', umbral=umbral, paso=paso,
    metricas_validacion=m_val, metricas_prueba=m_test,
    historial=resultado['historial'], mejor_epoca=resultado['mejor_epoca'],
    ruta_roc=ruta_roc)
print('entregables en', PERSISTENTE)
print(' -', ruta_md.name, '|', ruta_roc.name, '| reporte_metricas.json')
print(' -', pathlib.Path(resultado['ruta_pesos_cpu']).name, '(pesos para SP2)')